# Bit Manipulation — Technical Reference

*Reference — flat lookup, no prose. Scan or ctrl+F when you need a specific trick mid-problem.*

## Quick Index

| Technique | When to use | Problems |
| :--- | :--- | :--- |
| XOR tricks | Find single number, swap without temp | 136, 260 |
| AND tricks | Check bit, clear lowest set bit, power of two | 191, 231, 338 |
| Bit masking | Enumerate all subsets of a set | 78, 90 |
| Brian Kernighan | Count set bits in O(number of set bits) | 191, 338 |

## When to Use

| Signal | Technique to reach for |
| :--- | :--- |
| "appears once / twice / three times" | XOR tricks |
| "power of two", "count 1 bits", "hamming distance" | AND tricks |
| `n <= 20` and "all subsets" / "all assignments" | Bit masking |
| DP where the state is a *set* of visited items | Bitmask DP (state = integer) |
| "without using + or -", "without extra space" | Bitwise arithmetic |

---
## XOR Tricks

`a ^ a = 0` and `a ^ 0 = a`. XOR is commutative and associative — order doesn't matter. Use to find an element that appears an odd number of times, or to swap two values without a temp variable.

In [ ]:
# Swap without temp
a ^= b
b ^= a
a ^= b

# LC 136 — Single Number
# All numbers appear twice except one — XOR cancels pairs
def singleNumber(nums):
    return __import__('functools').reduce(lambda a, b: a ^ b, nums)

# LC 260 — Single Number III (two unique numbers)
# XOR all → result = a ^ b. Find any set bit → use as mask to separate a and b
def singleNumberIII(nums):
    xor = 0
    for n in nums: xor ^= n
    diff_bit = xor & (-xor)             # isolate rightmost set bit
    a = 0
    for n in nums:
        if n & diff_bit: a ^= n         # XOR only numbers with this bit set
    return [a, xor ^ a]

Problems: LC 136, LC 137, LC 260

---
## AND Tricks

Common bitwise operations using AND. Each trick is a one-liner.

In [ ]:
# Check if bit i is set
is_set = (n >> i) & 1

# Check odd / even
is_odd  = n & 1             # 1 if odd, 0 if even

# Clear the lowest set bit  — used in Brian Kernighan
n = n & (n - 1)

# Isolate the lowest set bit
lowest = n & (-n)

# Power of two check — exactly one bit set
is_power_of_two = n > 0 and (n & (n - 1)) == 0

# Set bit i
n |= (1 << i)

# Clear bit i
n &= ~(1 << i)

# Toggle bit i
n ^= (1 << i)

Problems: LC 231, LC 342, LC 191

---
## Bit Masking — Subset Enumeration

Represent each subset of `n` elements as an integer bitmask of `n` bits. Bit `i` is set if element `i` is included. Iterate from `0` to `2^n - 1` to enumerate all `2^n` subsets.

In [ ]:
# Enumerate all subsets of nums using bitmask
def subsets_bitmask(nums):
    n = len(nums)
    result = []
    for mask in range(1 << n):          # 1 << n = 2^n
        subset = []
        for i in range(n):
            if mask & (1 << i):         # bit i is set → include nums[i]
                subset.append(nums[i])
        result.append(subset)
    return result


# Enumerate all subsets of a bitmask (iterate over sub-masks)
# Used in DP on subsets problems
def enumerate_submasks(mask):
    sub = mask
    while sub > 0:
        # process sub
        sub = (sub - 1) & mask          # next smaller submask

Problems: LC 78, LC 90, LC 1986

---
## Brian Kernighan — Count Set Bits

`n & (n-1)` clears the lowest set bit. Repeat until `n == 0` — the number of iterations equals the number of set bits. Faster than checking each bit when the number is sparse.

In [ ]:
# Count set bits — O(number of set bits)
def countBits(n):
    count = 0
    while n:
        n &= n - 1                      # clear lowest set bit
        count += 1
    return count


# LC 338 — Counting Bits (count set bits for all numbers 0..n)
# DP: bits[i] = bits[i >> 1] + (i & 1)
# i >> 1 shifts right by 1 (same as i // 2); add 1 if i is odd
def countBitsDP(n):
    bits = [0] * (n + 1)
    for i in range(1, n + 1):
        bits[i] = bits[i >> 1] + (i & 1)
    return bits

Problems: LC 191, LC 338, LC 461

---
## Common mistakes

| Mistake | What you observe | Fix |
| :--- | :--- | :--- |
| Forgetting Python ints are arbitrary precision | Negative-number bit tricks behave unlike C/Java | Mask with `& 0xFFFFFFFF` when simulating 32-bit behaviour |
| `n & (n - 1) == 0` without checking `n > 0` | Reports 0 as a power of two | Guard with `n > 0 and (n & (n - 1)) == 0` |
| Operator precedence on `&` and `==` | Silently wrong condition | `&` binds *looser* than `==` in Python — parenthesise: `(n & 1) == 0` |
| Right-shifting a negative number expecting a logical shift | Loop never terminates — sign bit propagates forever | Mask to the width you want first |
| Using `1 << i` where `i` exceeds the intended width | Mask grows beyond the state space; wrong subset count | Bound `i` by `n` explicitly |
| Iterating `range(1 << n)` with n > 25 | TLE or memory blow-up | Bitmask enumeration is only viable for `n ≤ 20` |
| Confusing `^` with exponentiation | Wildly wrong numbers | `^` is XOR in Python; `**` is power |
| Toggling with `|` instead of `^` | Bit sets but never clears | `|=` sets, `&= ~` clears, `^=` toggles |